# From Graphs to Persistent Homology: The Algebraic Pipeline

This notebook walks through every algebraic object in the persistent homology pipeline — from a raw graph all the way to a persistence diagram — with runnable code at each step.

---
## 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import combinations
import networkx as nx
from ripser import ripser
from persim import plot_diagrams

---
## 1. Graphs and Cycles

A **graph** G = (V, E) is just vertices V and edges E.

**Cycle** — a closed loop: a path that starts and ends at the same vertex with no repeated edges.  
Analogy: a cycle is like a roundabout — you leave, drive around, and arrive back where you started.

The **cycle space** has dimension: `β₁ = |E| - |V| + connected_components`  
This is the **first Betti number** — it counts independent loops.

NetworkX lets us:
- build graphs as proper objects (not just coordinate arrays)
- compute the **cycle basis** algebraically with `nx.cycle_basis`
- use automatic layout algorithms so we never hardcode positions

We look at two graphs: a triangle (β₁ = 1) and a figure-eight (β₁ = 2).

In [ ]:
def draw_graph(G, ax, title, layout='spring', seed=0):
    """Draw a NetworkX graph with cycle-basis annotation."""
    if layout == 'spring':
        pos = nx.spring_layout(G, seed=seed)
    elif layout == 'circular':
        pos = nx.circular_layout(G)
    else:
        pos = layout   # allow passing a fixed dict

    cycles = nx.cycle_basis(G)
    beta1  = G.number_of_edges() - G.number_of_nodes() + nx.number_connected_components(G)

    nx.draw(G, pos, ax=ax,
            with_labels=True,
            labels={n: f'v{n}' for n in G.nodes()},
            node_color='steelblue', node_size=700,
            font_color='white', font_size=11,
            edge_color='black', width=2)

    ax.set_title(f'{title}\nβ₁ = {beta1}  |  cycle basis: {cycles}', fontsize=10)


fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Graph 1: Triangle  (β₁ = 1) ---
G_tri = nx.Graph()
G_tri.add_edges_from([(0,1),(1,2),(0,2)])
draw_graph(G_tri, axes[0], 'Triangle  (β₁ = 1)', layout='circular')

# --- Graph 2: Figure-eight / theta graph  (β₁ = 2) ---
# Two triangles sharing one vertex (v0)
G_eight = nx.Graph()
G_eight.add_edges_from([(0,1),(1,2),(0,2),   # left triangle
                         (0,3),(3,4),(0,4)])  # right triangle
pos_eight = {0:(0,0), 1:(-1,1), 2:(-1,-1), 3:(1,1), 4:(1,-1)}
draw_graph(G_eight, axes[1], 'Two triangles sharing v0  (β₁ = 2)', layout=pos_eight)

plt.tight_layout(); plt.show()

In [ ]:
# --- Highlight the cycle basis edges on the figure-eight ---
cycles = nx.cycle_basis(G_eight)
pos_eight = {0:(0,0), 1:(-1,1), 2:(-1,-1), 3:(1,1), 4:(1,-1)}

colors = ['crimson', 'darkorange']
fig, ax = plt.subplots(figsize=(5,4))

# Draw all edges in grey first
nx.draw_networkx_edges(G_eight, pos_eight, ax=ax, edge_color='lightgray', width=2)
nx.draw_networkx_nodes(G_eight, pos_eight, ax=ax, node_color='steelblue', node_size=700)
nx.draw_networkx_labels(G_eight, pos_eight, ax=ax,
                        labels={n: f'v{n}' for n in G_eight.nodes()},
                        font_color='white', font_size=11)

# Overlay each cycle in a distinct colour
for cycle, color in zip(cycles, colors):
    cycle_edges = list(zip(cycle, cycle[1:] + [cycle[0]]))
    nx.draw_networkx_edges(G_eight, pos_eight, edgelist=cycle_edges,
                           ax=ax, edge_color=color, width=3)

patches = [mpatches.Patch(color=c, label=f'Cycle {i+1}: {cyc}')
           for i,(cyc,c) in enumerate(zip(cycles, colors))]
ax.legend(handles=patches, loc='upper right', fontsize=9)
ax.set_title('Cycle basis highlighted  (β₁ = 2)', fontsize=11)
ax.axis('off')
plt.tight_layout(); plt.show()

print('Cycle basis:', cycles)
print(f'β₁ = |E|-|V|+components = {G_eight.number_of_edges()}-{G_eight.number_of_nodes()}+1 = {G_eight.number_of_edges()-G_eight.number_of_nodes()+1}')
print('Each coloured loop is one independent generator of the cycle space.')

---
## 2. Simplicial Complexes — Upgrading from Graphs

A graph only has 0-simplices (vertices) and 1-simplices (edges).  
A **simplicial complex** adds higher-dimensional pieces:

| Simplex | Dim | Analogy |
|---------|-----|---------|
| Vertex  | 0   | Dot |
| Edge    | 1   | Stick |
| Triangle (filled) | 2 | Tile |
| Tetrahedron (filled) | 3 | Brick |

Key rule: if a simplex is in the complex, **all its faces** must be too  
(e.g., a filled triangle requires all 3 edges and all 3 vertices).

A **Vietoris-Rips complex** at radius ε: add a k-simplex whenever all pairwise distances ≤ ε.  
Think of it as inflating bubbles around each point — when bubbles overlap, connect the points.

Below we build the complex manually for a small point cloud.

In [ ]:
pts = np.array([[0,0],[1,0],[0.5, np.sqrt(3)/2],[2,0]])

def vietoris_rips(points, epsilon):
    """Return simplices up to dim 2 for VR complex at radius epsilon."""
    n = len(points)
    simplices = {0: list(range(n)), 1: [], 2: []}
    for i,j in combinations(range(n), 2):
        if np.linalg.norm(points[i]-points[j]) <= epsilon:
            simplices[1].append((i,j))
    for i,j,k in combinations(range(n), 3):
        if all(np.linalg.norm(points[a]-points[b]) <= epsilon
               for a,b in [(i,j),(i,k),(j,k)]):
            simplices[2].append((i,j,k))
    return simplices

for eps in [0.8, 1.1, 2.1]:
    S = vietoris_rips(pts, eps)
    fig, ax = plt.subplots(figsize=(4,3.5))
    # filled triangles
    for (i,j,k) in S[2]:
        tri = plt.Polygon(pts[[i,j,k]], alpha=0.25, color='orange')
        ax.add_patch(tri)
    # edges
    for (i,j) in S[1]:
        ax.plot([pts[i,0],pts[j,0]],[pts[i,1],pts[j,1]],'k-',lw=2)
    # vertices
    ax.scatter(pts[:,0],pts[:,1],s=80,zorder=5,color='steelblue')
    ax.set_title(f'ε = {eps}  |  edges={len(S[1])}  triangles={len(S[2])}')
    ax.axis('equal'); ax.axis('off')
    plt.tight_layout(); plt.show()

---
## 3. Chain Groups — Algebra Enters

We now assign **algebra** to our geometry.  
Work over **𝔽₂** (integers mod 2: {0,1}).  Addition is XOR — no need to track orientation.

**k-chain**: a formal sum of k-simplices with coefficients in 𝔽₂.  
Analogy: a chain is like a shopping list — you tick off simplices you include (1) or skip (0).

The set of all k-chains forms the **chain group Cₖ**.  
For the triangle: C₀ = span{v0,v1,v2}, C₁ = span{e01,e12,e02}, C₂ = span{t012} (if filled).

Below we represent chains as binary vectors.

In [ ]:
# Triangle (hollow) simplices
V = ['v0','v1','v2']
E = ['e01','e12','e02']   # indexed as e_{min}_{max}

# A 1-chain: select edges e01 and e12  → vector [1,1,0]
chain_path = np.array([1,1,0], dtype=int)   # path v0-v1-v2
chain_loop  = np.array([1,1,1], dtype=int)  # full boundary loop

print('C₁ basis:', E)
print('Path chain (e01+e12):', chain_path)
print('Loop chain (e01+e12+e02):', chain_loop)
print()
print('Sum of path + one more edge e02 (mod 2):', (chain_path + np.array([0,0,1])) % 2)

---
## 4. Boundary Maps — Connecting Dimensions

The **boundary operator ∂ₖ : Cₖ → Cₖ₋₁** maps each simplex to its faces.  
Analogy: ∂ is like peeling the skin off a shape — an edge's boundary is its two endpoints; a triangle's boundary is its three edges.

Key rules (over 𝔽₂):
- ∂₁(edge [u,v]) = v + u   (mod 2, so just the two endpoints)
- ∂₂(triangle [u,v,w]) = [u,v] + [u,w] + [v,w]
- **∂∂ = 0** always — the boundary of a boundary is empty.

The boundary matrix **B** has shape `(dim k-1 simplices) × (dim k simplices)`.  
Entry B[i,j] = 1 if simplex i is a face of simplex j.

In [ ]:
# ---------- Boundary matrix for the hollow triangle ----------
#
#   Vertices:  v0, v1, v2
#   Edges:     e01=[v0,v1], e12=[v1,v2], e02=[v0,v2]

# ∂₁ : C₁ → C₀   rows=vertices, cols=edges
#          e01  e12  e02
d1 = np.array([
    [1,   0,   1],   # v0
    [1,   1,   0],   # v1
    [0,   1,   1],   # v2
], dtype=int)

print('∂₁  (rows=vertices, cols=edges):')
print(d1)
print()

# Verify ∂₁ on the full loop [1,1,1] → should be zero (mod 2)
loop = np.array([1,1,1])
boundary_of_loop = d1 @ loop % 2
print('∂₁(e01+e12+e02) =', boundary_of_loop, '← zero vector = it is a cycle ✓')

# Verify ∂∂ = 0 — only meaningful if we have a filled triangle
# Add the filled triangle; ∂₂ maps it to the sum of its edges
d2 = np.array([[1],[1],[1]], dtype=int)   # one 2-simplex
dd = d1 @ d2 % 2
print()
print('∂₁ ∘ ∂₂ (should be zero):', dd.T, '← ∂∂=0 verified ✓')

---
## 5. Cycles, Boundaries, and Homology

From the boundary maps we extract:

| Object | Definition | Meaning |
|--------|-----------|--------|
| **Cycle group Zₖ** | kernel of ∂ₖ — chains with no boundary | closed loops |
| **Boundary group Bₖ** | image of ∂ₖ₊₁ — boundaries of (k+1)-chains | loops that are filled |
| **Homology Hₖ** | Zₖ / Bₖ — cycles *not* filled in | "holes" in dimension k |
| **Betti number βₖ** | dim(Hₖ) | count of independent k-holes |

Analogy: a cycle is a rubber band laid on the space.  
A boundary is a rubber band you can shrink to a point (it bounds a filled region).  
Homology counts the rubber bands that *cannot* be shrunk — they go around real holes.

For a **hollow triangle**:
- β₀ = 1 (one connected component)
- β₁ = 1 (the loop around the hole — no filled triangle to kill it)

For a **filled triangle**:
- β₀ = 1
- β₁ = 0 (the triangle fills in the loop → it becomes a boundary → trivial in homology)

In [ ]:
def rank_F2(M):
    """Gaussian elimination over F2, return rank."""
    A = M.copy() % 2
    rows, cols = A.shape
    pivot_row = 0
    for col in range(cols):
        found = -1
        for r in range(pivot_row, rows):
            if A[r, col] == 1:
                found = r
                break
        if found == -1:
            continue
        A[[pivot_row, found]] = A[[found, pivot_row]]
        for r in range(rows):
            if r != pivot_row and A[r, col] == 1:
                A[r] = (A[r] + A[pivot_row]) % 2
        pivot_row += 1
    return pivot_row

# --- Hollow triangle ---
# Trivial ∂₂ (no 2-simplices)
d2_hollow = np.zeros((3, 0), dtype=int)   # no triangles

dim_Z1 = 3 - rank_F2(d1)                  # dim(ker ∂₁)
dim_B1 = 0                                 # dim(im ∂₂) — no triangles
beta1_hollow = dim_Z1 - dim_B1
print('=== Hollow triangle ===')
print(f'dim(Z₁) = {dim_Z1},  dim(B₁) = {dim_B1},  β₁ = {beta1_hollow}')

# --- Filled triangle ---
d2_filled = np.array([[1],[1],[1]], dtype=int)
dim_B1_filled = rank_F2(d2_filled)
beta1_filled = dim_Z1 - dim_B1_filled
print()
print('=== Filled triangle ===')
print(f'dim(Z₁) = {dim_Z1},  dim(B₁) = {dim_B1_filled},  β₁ = {beta1_filled}')

---
## 6. Filtrations — Homology Over Time

So far we have a **static** complex. Persistent homology adds **time** (the filtration parameter ε).

A **filtration** is a nested sequence of complexes:  
`∅ = K₀ ⊆ K₁ ⊆ K₂ ⊆ … ⊆ Kₙ = K`

Analogy: imagine slowly filling a room with water. Features (islands = components, pools = loops) appear and disappear as the water rises. Filtration value = water level.

For a point cloud with pairwise distances, we grow ε from 0 to max distance, adding edges and higher simplices as they become eligible.

Each homology class gets a **birth** (ε where it first appears) and a **death** (ε where it gets filled in).  
**Persistence = death − birth** — how long the feature survives.

Below we animate the filtration of 5 points.

In [ ]:
np.random.seed(42)
pts5 = np.array([[0,0],[1,0],[0.5,0.87],[2,0.3],[1.5,0.9]])

dists = np.array([[np.linalg.norm(pts5[i]-pts5[j])
                   for j in range(5)] for i in range(5)])

eps_vals = np.linspace(0, 2.2, 6)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

for ax, eps in zip(axes.flat, eps_vals):
    S = vietoris_rips(pts5, eps)
    for (i,j,k) in S[2]:
        tri = plt.Polygon(pts5[[i,j,k]], alpha=0.2, color='orange')
        ax.add_patch(tri)
    for (i,j) in S[1]:
        ax.plot([pts5[i,0],pts5[j,0]],[pts5[i,1],pts5[j,1]],'k-',lw=1.5)
    ax.scatter(pts5[:,0],pts5[:,1],s=60,zorder=5,color='steelblue')
    ax.set_title(f'ε = {eps:.2f}  |  edges={len(S[1])}  tri={len(S[2])}')
    ax.axis('equal'); ax.axis('off')

plt.suptitle('Vietoris-Rips filtration — growing ε', fontsize=13)
plt.tight_layout(); plt.show()

---
## 7. Persistence Algorithm — Reduction of Boundary Matrices

For each filtration step we have a boundary matrix.  
The **persistence algorithm** is column-reduction (Gaussian elimination) of the combined boundary matrix, ordered by simplex birth time.

**Column reduction rule** (over 𝔽₂):  
If two columns share the same lowest 1-entry (pivot), add the left column to the right.
Repeat until no two columns share a pivot.

Output pairing:
- A **zero column** for simplex σ → σ is a cycle; birth of a new class.
- A **nonzero column** for simplex τ with pivot at row σ → σ-τ pair; class born at σ dies at τ.

Analogy: the algorithm is like a matchmaker — it pairs each "death" simplex with the "birth" simplex whose hole it fills.

Below we implement reduction for a tiny example by hand.

In [ ]:
def reduce_boundary_matrix(D):
    """Column-reduce boundary matrix D over F2.
    Returns (R, V) where D = R (reduced) and pairing info."""
    R = D.copy() % 2
    m = R.shape[1]
    pivot_col = {}   # pivot_row -> col index

    def low(col):
        nz = np.where(R[:, col] == 1)[0]
        return nz[-1] if len(nz) > 0 else -1

    for j in range(m):
        while True:
            lj = low(j)
            if lj == -1:
                break
            if lj not in pivot_col:
                pivot_col[lj] = j
                break
            R[:, j] = (R[:, j] + R[:, pivot_col[lj]]) % 2

    # Build pairs: (birth_simplex, death_simplex)
    pairs = []
    for j in range(m):
        lj = low(j)
        if lj != -1:
            pairs.append((lj, j))

    return R, pairs


# --- Small worked example: 4 simplices ---
# Simplices ordered by birth:  v0(0), v1(1), e01(2), e12(3)
# (using a path v0-v1-v2 with v2 shared as v1 reused — simplified)
#
# ∂:  v0  v1  e01  e12
#      0   0    1    0    <- v0
#      0   0    1    1    <- v1
#      0   0    0    1    <- v2  (pretend third vertex)

D = np.array([
    [0, 0, 1, 0],
    [0, 0, 1, 1],
    [0, 0, 0, 1],
], dtype=int)

print('Boundary matrix D:')
print(D)
R, pairs = reduce_boundary_matrix(D)
print('\nReduced matrix R:')
print(R)
print('\nPaired (birth_row, death_col) indices:', pairs)
print('→ Simplex 1 (v1) and simplex 3 (e12) pair: H₀ class born at v1, killed when e12 connects it.')

---
## 8. Persistence Diagrams — Reading the Output

Each pair (birth b, death d) becomes a **point** in the persistence diagram.  
- **H₀ points** — connected components: born when a new vertex appears, die when an edge merges two components.
- **H₁ points** — loops: born when an edge closes a cycle, die when a triangle fills it.

Distance from the diagonal = persistence = signal strength.  
Points near the diagonal = noise (born and die quickly).  
Points far from the diagonal = genuine topology.

Now we run **Ripser** (which does the full reduction efficiently) on a noisy circle and read the diagram.

In [ ]:
np.random.seed(7)
n = 150
theta = np.linspace(0, 2*np.pi, n)
X = np.column_stack([np.cos(theta), np.sin(theta)])
X += np.random.normal(0, 0.1, X.shape)

dgms = ripser(X, maxdim=1)['dgms']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Point cloud
axes[0].scatter(X[:,0], X[:,1], s=15, color='steelblue', alpha=0.7)
axes[0].set_title('Noisy circle  (n=150, noise=0.1)')
axes[0].set_aspect('equal')

# Persistence diagram
plot_diagrams(dgms, ax=axes[1], title='Persistence diagram')
plt.tight_layout(); plt.show()

h1 = dgms[1]
finite = h1[h1[:,1] < np.inf]
persistence = finite[:,1] - finite[:,0]
idx = np.argmax(persistence)
b, d = finite[idx]
print(f'H₁ most persistent bar:  birth={b:.3f}  death={d:.3f}  persistence={d-b:.3f}')
print('→ One long bar = one loop = the circle.')

---
## 9. Full Pipeline Summary

```
Point cloud  (metric space)
      │
      ▼  grow ε
Filtration of Vietoris-Rips complexes
      │
      ▼  formal sums mod 2
Chain groups  Cₖ
      │
      ▼  boundary maps ∂ₖ
Boundary matrices  Bₖ
      │
      ▼  column reduction (Gaussian elim over F2)
Reduced matrices  →  birth/death pairs
      │
      ▼
Persistence diagram  (H₀, H₁, H₂, …)
      │
      ▼  threshold by persistence
Topological signal  (Betti numbers, features for ML)
```

**The one-sentence version:**  
PH tracks when algebraic holes (elements of Hₖ = Zₖ/Bₖ) are created and destroyed as you grow a simplicial complex — boundary matrix reduction is the engine that pairs creators with destroyers.

In [ ]:
# --- Barcode for H₁: visualise birth/death intervals ---
h1_finite = dgms[1][dgms[1][:,1] < np.inf]
sorted_idx = np.argsort(h1_finite[:,1] - h1_finite[:,0])[::-1]
h1_sorted = h1_finite[sorted_idx]

fig, ax = plt.subplots(figsize=(9,4))
for i, (b,d) in enumerate(h1_sorted):
    color = 'crimson' if (d-b) == max(h1_sorted[:,1]-h1_sorted[:,0]) else 'steelblue'
    ax.plot([b, d], [i, i], lw=3 if color=='crimson' else 1.5, color=color, alpha=0.85)
ax.set_xlabel('Filtration value (ε)')
ax.set_title('H₁ Barcode — red bar = the circle, blue = noise')
ax.set_yticks([])
red_patch = mpatches.Patch(color='crimson', label='Signal (circle)')
blue_patch = mpatches.Patch(color='steelblue', label='Noise')
ax.legend(handles=[red_patch, blue_patch], loc='lower right')
plt.tight_layout(); plt.show()